In [1]:
# Tecnología
import json
import calendar
import pandas as pd
from sparky_bc import Sparky
import datetime as dt
from dateutil.relativedelta import relativedelta

# files lz conection
path_sparky_conf = '/Users/santlond/Documents/sparky_conf.json'

# Configurar conexión a LZ
with open(path_sparky_conf, 'rb') as JSON_lz_File:
    sp_config = json.loads(JSON_lz_File.read())
    
USER='santlond'
PASS=sp_config['ID']
DSN='IMPALA_PROD'
LOGDIR= 'logs'
# sparky = Sparky(username=USER, password=PASS, dsn=DSN, hostname="sbmdeblze004.bancolombia.corp")
sparky = Sparky(username=USER, password=PASS, dsn=DSN, hostname="sbmdeblze004.bancolombia.corp", spark_submit="spark3-submit")
 
# sparky = Sparky(username=USER, password=PASS, dsn=DSN)

helper = sparky.helper

/Users/santlond/Documents/venv_py39_odbc/lib/python3.9/site-packages/helper/helper.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2026-06-15 18:16:38 - [WARNING] - No se encontro la carpeta "/Users/santlond/Documents/ADQUIRENCIA_FERIA_EVA/logs" para guardar los logs


 ____  _____ __  __  ___ _____ _____ 
|  _ \| ____|  \/  |/ _ \_   _| ____|
| |_) |  _| | |\/| | | | || | |  _|  
|  _ <| |___| |  | | |_| || | | |___ 
|_| \_\_____|_|  |_|\___/ |_| |_____|
                                     
 ____  ____   _    ____  _  __
/ ___||  _ \ / \  |  _ \| |/ /
\___ \| |_) / _ \ | |_) | ' / 
 ___) |  __/ ___ \|  _ <| . \ 
|____/|_| /_/   \_\_| \_\_|\_\
                              



# Evolución vinculación wompi vinculacion completa

In [2]:
sql_drop = """DROP TABLE IF EXISTS proceso.mdo_wompi_vinc_completa_vinculaciones_hist PURGE;"""
helper.ejecutar_consulta(sql_drop)

sql = """
CREATE TABLE proceso.mdo_wompi_vinc_completa_vinculaciones_hist STORED AS PARQUET AS WITH news AS
  (SELECT periodo,
          num_vinc AS num_vinc_new,
          cast(left(cast(periodo AS STRING), 4) AS int) AS YEAR,
          cast(right(cast(periodo AS STRING), 2) AS int) AS mes,
          1 AS secuencia
   FROM proceso_vdm.mdo_wompi_vinc_completa_vinculaciones_hist
   WHERE tipo_cliente = 'nuevos' ),
                                                                             news_out AS
  (SELECT periodo,
          num_vinc_new,
          sum(num_vinc_new) OVER (PARTITION BY YEAR
                                  ORDER BY YEAR, mes) AS num_vinc_new_cumsum_ym,
          sum(num_vinc_new) OVER (PARTITION BY secuencia
                                  ORDER BY periodo) AS num_vinc_new_cumsum
   FROM news),
                                                                             olds AS
  (SELECT periodo,
          num_vinc AS num_vinc_old
   FROM proceso_vdm.mdo_wompi_vinc_completa_vinculaciones_hist
   WHERE tipo_cliente = 'viejos' ),
                                                                             alls AS
  (SELECT periodo,
          num_vinc AS num_vinc_all
   FROM proceso_vdm.mdo_wompi_vinc_completa_vinculaciones_hist
   WHERE tipo_cliente = 'todos' )
SELECT n.periodo,
       n.num_vinc_new,
       n.num_vinc_new_cumsum_ym,
       n.num_vinc_new_cumsum,
       o.num_vinc_old,
       a.num_vinc_all
FROM news_out AS n
LEFT JOIN olds AS o ON n.periodo = o.periodo
LEFT JOIN alls AS a ON n.periodo = a.periodo
ORDER BY periodo DESC
"""
helper.ejecutar_consulta(sql)

sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_wompi_vinc_completa_vinculaciones_hist;"""
helper.ejecutar_consulta(sql_compute)

2026-06-15 18:16:39 - [INFO] - Transcurrido: 1781565400, Tiempo de Refresco = 1000


------------------------------------------------------------------------------------------
  i  tipo                  nombre                     estado     hora_inicio   duracion   
------------------------------------------------------------------------------------------
 1/1 DROP ...ompi_vinc_completa_vinculaciones_hist   finalizado   06:16:40 PM     00:00.4 
------------------------------------------------------------------------------------------
--------------------------------------------------------------------------------------------
  i   tipo                   nombre                     estado     hora_inicio   duracion   
--------------------------------------------------------------------------------------------
 2/2 CREATE ...ompi_vinc_completa_vinculaciones_hist   finalizado   06:16:40 PM     00:01.4 
--------------------------------------------------------------------------------------------
--------------------------------------------------------------------------------

# Uso de vinculados wompi vinculacion completa

In [3]:
sql_drop = """DROP TABLE IF EXISTS proceso.mdo_hist_vinc_uso_wompi_vinc_completa PURGE;"""
helper.ejecutar_consulta(sql_drop)

sql = """
CREATE TABLE proceso.mdo_hist_vinc_uso_wompi_vinc_completa STORED AS PARQUET AS WITH news AS
  (SELECT periodo,
          count(*) AS num_vinc_new_uso_cumsum_ym
   FROM proceso_vdm.mdo_wompi_vinc_completa_vinculaciones_con_trxs_hist
   WHERE tipo_cliente = 'nuevos'
   GROUP BY 1),
                                                                                 olds AS
  (SELECT periodo,
          count(*) AS num_vinc_old_uso_cumsum_ym
   FROM proceso_vdm.mdo_wompi_vinc_completa_vinculaciones_con_trxs_hist
   WHERE tipo_cliente = 'viejos'
   GROUP BY 1),
                                                                                 alls AS
  (SELECT periodo,
          count(*) AS num_vinc_all_uso_cumsum_ym
   FROM proceso_vdm.mdo_wompi_vinc_completa_vinculaciones_con_trxs_hist
   WHERE tipo_cliente = 'todos'
   GROUP BY 1)
SELECT n.periodo,
       n.num_vinc_new_uso_cumsum_ym,
       o.num_vinc_old_uso_cumsum_ym,
       a.num_vinc_all_uso_cumsum_ym
FROM news AS n
LEFT JOIN olds AS o ON n.periodo = o.periodo
LEFT JOIN alls AS a ON n.periodo = a.periodo
ORDER BY n.periodo DESC;
"""
helper.ejecutar_consulta(sql)

sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_hist_vinc_uso_wompi_vinc_completa;"""
helper.ejecutar_consulta(sql_compute)

---------------------------------------------------------------------------------------------
  i   tipo                    nombre                     estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------
 4/4    DROP ...mdo_hist_vinc_uso_wompi_vinc_completa   finalizado   06:16:43 PM     00:00.2 
---------------------------------------------------------------------------------------------
---------------------------------------------------------------------------------------------
  i   tipo                    nombre                     estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------
 5/5  CREATE ...mdo_hist_vinc_uso_wompi_vinc_completa   finalizado   06:16:43 PM     00:01.5 
---------------------------------------------------------------------------------------------
------------------------------------------------------------

# Tabla resultado

In [4]:
# sql = """
# WITH outcome1 AS
#   (SELECT a.fecha_ym,
#           a.num_vinc_new,
#           a.num_vinc_cumsum,
#           a.num_vinc_new_cumsum_ym,
#           nvl(b.num_vinc_new_uso_cumsum_ym, 0) AS num_vinc_new_uso_cumsum_ym,
#           round(nvl(b.num_vinc_new_uso_cumsum_ym, 0)/a.num_vinc_new_cumsum_ym, 4) AS num_vinc_new_prop_uso,
#           nvl(c.num_vinc_old_uso_cumsum_ym, 0) AS num_vinc_old_uso_cumsum_ym,
#           nvl(d.num_vinc_all_uso_cumsum_ym, 0) AS num_vinc_all_uso_cumsum_ym,
#           round(nvl(d.num_vinc_all_uso_cumsum_ym, 0)/a.num_vinc_cumsum, 4) AS num_vinc_all_prop_uso,
#           left(cast(a.fecha_ym AS string), 4) AS YEAR,
#           right(cast(a.fecha_ym AS string), 2) AS mes
#    FROM proceso.mdo_aceptacion_comercios_vinc AS a
#    LEFT JOIN proceso.mdo_aceptacion_comercios_num_vinc_new_uso AS b ON a.fecha_ym = b.fecha_ym
#    LEFT JOIN proceso.mdo_aceptacion_comercios_num_vinc_old_uso AS c ON a.fecha_ym = c.fecha_ym
#    LEFT JOIN proceso.mdo_aceptacion_comercios_num_vinc_all_uso AS d ON a.fecha_ym = d.fecha_ym),
#      outcome2 AS
#   (SELECT fecha_ym,
#           num_vinc_cumsum AS num_vinc_old,
#           cast(cast(YEAR AS int) + 1 AS string) AS YEAR
#    FROM outcome1
#    WHERE mes = '12')
# SELECT a.fecha_ym,
#        CONCAT(a.YEAR, '/', a.mes, '/', '01') AS fecha_ym2,
#        a.num_vinc_new,
#        a.num_vinc_new_cumsum_ym,
#        a.num_vinc_new_uso_cumsum_ym,
#        a.num_vinc_new_prop_uso,
#        b.num_vinc_old,
#        a.num_vinc_old_uso_cumsum_ym,
#        round(a.num_vinc_old_uso_cumsum_ym/b.num_vinc_old, 4) AS num_vinc_old_prop_uso,
#        a.num_vinc_cumsum,
#        a.num_vinc_all_uso_cumsum_ym,
#        a.num_vinc_all_prop_uso
# FROM outcome1 AS a
# LEFT JOIN outcome2 AS b ON a.year = b.year
# WHERE a.fecha_ym BETWEEN 202201 AND 202511
# ORDER BY a.fecha_ym DESC;
# """
# # print(sql)
# df_outcome = helper.obtener_dataframe(sql)
# df_outcome

In [5]:
sql_drop = """DROP TABLE IF EXISTS proceso_vdm.mdo_hist_vinc_y_uso_wompi_vinc_completa PURGE;"""
helper.ejecutar_consulta(sql_drop)

sql = """
CREATE TABLE proceso_vdm.mdo_hist_vinc_y_uso_wompi_vinc_completa STORED AS PARQUET AS
SELECT a.periodo,
       concat(cast(a.periodo as string), '01') AS fecha_ymd2,
       a.num_vinc_new,
       a.num_vinc_new_cumsum_ym,
       a.num_vinc_new_cumsum,
       a.num_vinc_old,
       a.num_vinc_all,
       b.num_vinc_new_uso_cumsum_ym,
       b.num_vinc_old_uso_cumsum_ym,
       b.num_vinc_all_uso_cumsum_ym,
       round(b.num_vinc_new_uso_cumsum_ym/a.num_vinc_new_cumsum_ym, 4) AS num_vinc_new_prop_uso,
       round(b.num_vinc_old_uso_cumsum_ym/a.num_vinc_old, 4) AS num_vinc_old_prop_uso,
       round(b.num_vinc_all_uso_cumsum_ym/a.num_vinc_all, 4) AS num_vinc_all_prop_uso
FROM proceso.mdo_wompi_vinc_completa_vinculaciones_hist AS a
LEFT JOIN proceso.mdo_hist_vinc_uso_wompi_vinc_completa AS b ON a.periodo = b.periodo
ORDER BY a.periodo DESC;
"""
helper.ejecutar_consulta(sql)

sql_compute = """COMPUTE INCREMENTAL STATS proceso_vdm.mdo_hist_vinc_y_uso_wompi_vinc_completa;"""
helper.ejecutar_consulta(sql_compute)

---------------------------------------------------------------------------------------------
  i   tipo                    nombre                     estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------
 7/7    DROP ...o_hist_vinc_y_uso_wompi_vinc_completa   finalizado   06:16:47 PM     00:00.2 
---------------------------------------------------------------------------------------------
---------------------------------------------------------------------------------------------
  i   tipo                    nombre                     estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------
 8/8  CREATE ...o_hist_vinc_y_uso_wompi_vinc_completa   finalizado   06:16:47 PM     00:00.9 
---------------------------------------------------------------------------------------------
------------------------------------------------------------

In [6]:
sql = """
SELECT a.periodo,
       a.fecha_ymd2,
       a.num_vinc_new,
       a.num_vinc_new_cumsum_ym,
       a.num_vinc_new_cumsum,
       a.num_vinc_old,
       a.num_vinc_all,
       a.num_vinc_new_uso_cumsum_ym,
       a.num_vinc_old_uso_cumsum_ym,
       a.num_vinc_all_uso_cumsum_ym,
       a.num_vinc_new_prop_uso,
       a.num_vinc_old_prop_uso,
       a.num_vinc_all_prop_uso
FROM proceso_vdm.mdo_hist_vinc_y_uso_wompi_vinc_completa AS a
ORDER BY a.periodo DESC;
"""
df_outcome = helper.obtener_dataframe(sql)
df_outcome

-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 10/10 DATAFRAME                                           descargando   06:16:48 PM             

2026-06-15 18:16:49 - [INFO] - 44 filas, 13 columnas, 00:00.6 consultando, 00:00.1 descargando, 00:00.0 convirtiendo


 10/10 DATAFRAME                                            finalizado   06:16:48 PM     00:00.9 
-------------------------------------------------------------------------------------------------


,periodo,fecha_ymd2,num_vinc_new,num_vinc_new_cumsum_ym,num_vinc_new_cumsum,num_vinc_old,num_vinc_all,num_vinc_new_uso_cumsum_ym,num_vinc_old_uso_cumsum_ym,num_vinc_all_uso_cumsum_ym,num_vinc_new_prop_uso,num_vinc_old_prop_uso,num_vinc_all_prop_uso
0,202604.0,20260401,1,2,16760,39685,39687,1.0,6297.0,6299.0,0.5000,0.1587,0.1587
1,202603.0,20260301,1,1,16759,39685,39686,1.0,5933.0,5934.0,1.0000,0.1495,0.1495
2,202506.0,20250601,1,15,16758,39670,39685,NaN,NaN,NaN,NaN,NaN,NaN
3,202505.0,20250501,2,14,16757,39670,39684,1.0,7583.0,7590.0,0.0714,0.1912,0.1913
4,202504.0,20250401,1,12,16755,39670,39682,NaN,NaN,NaN,NaN,NaN,NaN
5,202503.0,20250301,4,11,16754,39670,39681,1.0,6796.0,6802.0,0.0909,0.1713,0.1714
6,202502.0,20250201,5,7,16750,39670,39677,3.0,6193.0,6198.0,0.4286,0.1561,0.1562
7,202501.0,20250101,2,2,16745,39670,39672,NaN,NaN,NaN,NaN,NaN,NaN
8,202412.0,20241201,3,530,16743,39140,39670,2.0,10333.0,10623.0,0.0038,0.2640,0.2678
9,202411.0,20241101,7,527,16740,39140,39667,6.0,10173.0,10459.0,0.0114,0.2599,0.2637


In [7]:
df_outcome.to_excel('main_data/evolucion_vinculacion_y_uso_wompi_vinc_completa.xlsx', index=False)

# Eliminación tablas proceso.

In [8]:
# Eliminación tablas proceso.
tablas_borrar = ['proceso.mdo_wompi_vinc_completa_vinculaciones_hist', 'proceso.mdo_hist_vinc_uso_wompi_vinc_completa']

for tabla in tablas_borrar:
    sql_drop = f"""DROP TABLE IF EXISTS {tabla} PURGE;"""
    helper.ejecutar_consulta(sql_drop)

-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 11/11      DROP ...ompi_vinc_completa_vinculaciones_hist   finalizado   06:16:50 PM     00:00.2 
-------------------------------------------------------------------------------------------------
-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 12/12      DROP ...mdo_hist_vinc_uso_wompi_vinc_completa   finalizado   06:16:50 PM     00:00.2 
-------------------------------------------------------------------------------------------------
